# Wavefront reconstruction from lantern signals at one wavelength

Reconstructs the first 5 DM modes (the petal pistons of the bundled ANDES
modal basis) from the output-core signals of the photonic lantern at its
design wavelength, with an MLP and with explainable alternatives: a linear
reconstructor (optionally with symbolic-regression corrections) and a sparse
cubic (OMP) polynomial of the core signals.

The coefficients are uniform in +-span, inside the +-lambda/2 range where a
single piston is unambiguous at one wavelength (+-lambda/2: 400 nm at 800 nm,
600 nm at 1200 nm; see `trainMLP_multiwavelength.ipynb` / `PL_EML.ipynb` for larger
pistons).

**Why the port count matters at one wavelength.** Each single piston is kept
inside +-lambda/2, but two petals can differ by twice that. The signals of an
N-port lantern, normalised, have N - 1 independent values; a smooth map of the
5 pistons is in general one-to-one over a large range only into 11 or more
dimensions, otherwise it folds onto itself and different piston sets give the
same signals. 19 ports (18 values) resolve the range; 7 ports (6 values) do
not: on `pl7_1200` every reconstructor stops far above the 19-port lanterns,
while the same lantern reconstructs smaller pistons well. At exactly +-lambda/2
a piston equals its opposite (+lambda/2 and -lambda/2 give the same field), so
the edges of the range are ambiguous for every lantern.
`pl19_1200` (same fibres and wavelength, 19 ports) is the direct test.

**Lantern.** The notebook runs for the lantern preset named by the environment
variable `CBEAM_LANTERN` (cbeam's `batch_pipeline.LANTERNS`): `pl19_800`, the
reference 19-port lantern at 800 nm (default); `pl7_1200`, 7 ports at
1200 nm; `pl19_1200`, 19 ports at 1200 nm in the same fibres as `pl7_1200`.
Its settings (piston range, polynomial budgets, data folder) are in
`ml_reconstruct.presets`; executed copies for each lantern are in
`runs/<lantern>/`.

Fitted models are cached under the lantern's data folder (`retrain=True` in
the cache below refits them).


In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"

import numpy as np
import matplotlib.pyplot as plt
import ml_reconstruct as ml

from ml_reconstruct.presets import settings
S = settings()                  # this lantern's settings (CBEAM_LANTERN)
LANTERN, DATA = S["lantern"], S["data_dir"]
N_SAMPLES, N_TRAIN = 10000, 8000
M_MODES = 5                     # DM modes 0..4 (petal pistons)
AMPLITUDE_NM = S["single_span_nm"]
DATASET = S["single"]
# phasor-target models (see below); cache_single keeps the earlier direct-piston fits
cache = ml.ModelCache(f"{DATA}/cache_single_phasor", retrain=False)
print(f"lantern {LANTERN}: pistons +-{AMPLITUDE_NM:.0f} nm, data in {DATA}")

## Dataset

Each sample is a random configuration of the 5 modes, propagated through
the lantern; the signal of each core is the mean of a 5x5-pixel window of
the imaged output face around the core centre, with the input normalised to
unit coupled power (see `ml_reconstruct.simulate`). Simulating 10000 samples
takes 15-40 minutes on the numpy backend (`gen_datasets.py single`).

In [ ]:
if os.path.exists(DATASET):
    ds = ml.LanternDataset.load(DATASET)
else:
    from ml_reconstruct.simulate import bp, simulate_core_images   # bp: cbeam's batch_pipeline
    p = bp.get_simulation_parameters(lantern=LANTERN)
    ifunc = bp.load_ifunc(p["ifunc_file"]); bp.center_ifunc_mask(ifunc)
    coeff, _ = bp.create_random_aberration_configs(
        N_SAMPLES, M_MODES, -AMPLITUDE_NM, AMPLITUDE_NM, rng=0)
    ds = simulate_core_images(p, ifunc, coeff, N_TRAIN)
    os.makedirs(os.path.dirname(DATASET), exist_ok=True)
    ds.save(DATASET)

S_tr, C_tr = ds.train
S_va, C_va = ds.val
# At one wavelength the signals depend on each piston only modulo lambda: p and p +- lambda give the
# same signals. Regressing p itself asks every model to reproduce the jump of the sawtooth at +-lambda/2;
# the models regress instead the phasor (sin, cos)(2 pi p / lambda) of each piston (2 outputs per piston),
# which has no jump, and the estimate is decoded with atan2 into [-lambda/2, lambda/2).
LAMBDA = 2 * AMPLITUDE_NM       # the range is +-lambda/2 at the design wavelength
xs, ys = ml.Standardizer().fit(S_tr), ml.PhasorCodec(LAMBDA)        # inputs: training split only
Xtr, Xva, Ytr = xs.transform(S_tr), xs.transform(S_va), ys.transform(C_tr)
print(f"{len(S_tr)} training / {len(S_va)} validation samples, {S_tr.shape[1]} cores, "
      f"{C_tr.shape[1]} modes, coefficients std {C_tr.std():.0f} nm")

RESULTS = {}                    # model -> per-mode validation RMSE [nm]
WRAPPED = {}                    # the same, with the errors taken modulo lambda: the figure of merit
def record(name, Y_pred_std):
    """Score predicted phasor targets against the validation pistons."""
    RESULTS[name] = ml.rmse_nm(ys.inverse(Y_pred_std), C_va)
    WRAPPED[name] = ml.rmse_nm(ys.inverse(Y_pred_std), C_va, LAMBDA)
    print(f"{name:16s} mean {RESULTS[name].mean():6.1f} nm   per mode {np.round(RESULTS[name], 1)}"
          f"   modulo lambda {WRAPPED[name].mean():6.1f} nm")

## MLP

Three hidden layers of 128 units, Adam, 500 epochs; the weights of the epoch
with the lowest validation loss are kept.

In [ ]:
LAYERS = [Xtr.shape[1], 128, 128, 128, Ytr.shape[1]]     # (sin, cos) of each piston
MLP_FILE = f"{DATA}/cache_single_phasor/mlp-{'x'.join(map(str, LAYERS))}-e500.npz"
if os.path.exists(MLP_FILE) and not cache.retrain:
    params, _, extra = ml.load_mlp(MLP_FILE)
    hist = {"train": extra["train"], "val": extra["val"], "best_epoch": int(extra["best_epoch"])}
else:
    params, hist = ml.train_mlp(LAYERS, Xtr, Ytr, Xva, ys.transform(C_va),
                                epochs=500, learning_rate=3e-4)
    os.makedirs(os.path.dirname(MLP_FILE), exist_ok=True)
    ml.save_mlp(MLP_FILE, params, LAYERS, train=hist["train"], val=hist["val"],
                best_epoch=hist["best_epoch"])
print(f"best epoch {hist['best_epoch']}")
record("MLP", np.asarray(ml.mlp_forward(params, Xva)))

C_mlp = ys.inverse(np.asarray(ml.mlp_forward(params, Xva)))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["train"], label="train"); ax[0].plot(hist["val"], label="validation")
ax[0].set_yscale("log"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("Huber loss"); ax[0].legend()
k = 2
ax[1].scatter(C_va[:, k], C_mlp[:, k], s=4, alpha=0.4)
ax[1].plot([-AMPLITUDE_NM, AMPLITUDE_NM], [-AMPLITUDE_NM, AMPLITUDE_NM], "r--")
ax[1].set_xlabel("true piston (nm)"); ax[1].set_ylabel("MLP estimate (nm)")
ax[1].set_title(f"piston {k}")
plt.tight_layout(); plt.show()

## Baselines: linear reconstructor and k-nearest neighbours

The linear reconstructor is a 5 x n_cores matrix (ridge-regularised); k-NN needs no
training and shows how well the signals determine the coefficients at all.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor

linear = cache("linear", lambda: Ridge(1.0).fit(Xtr, Ytr))
record("linear", linear.predict(Xva))
knn = cache("kNN", lambda: KNeighborsRegressor(5, weights="distance").fit(Xtr, Ytr))
record("kNN", knn.predict(Xva))

## Optional: linear reconstructor + symbolic-regression residual

PySR searches, per mode, for a compact formula of the core signals that
explains what the linear reconstructor misses. It needs PySR (and Julia),
and only improves the linear model marginally, so it is off by default.

In [ ]:
RUN_SR = False

def _inv(x):                    # module-level, so the fitted model can be pickled
    return 1 / x

def fit_linear_sr():
    from pysr import PySRRegressor
    residual = Ytr - linear.predict(Xtr)
    models = []
    for k in range(M_MODES):
        m = PySRRegressor(
            niterations=100, populations=41, population_size=60, maxsize=35,
            binary_operators=["+", "-", "*"], unary_operators=["square", "inv(x) = 1/x", "sin"],
            elementwise_loss="L2DistLoss()", parsimony=0.0, model_selection="best",
            parallelism="multithreading", turbo=True, random_state=0, verbosity=0,
            progress=False, extra_sympy_mappings={"inv": _inv})
        models.append(m.fit(Xtr, residual[:, k], variable_names=[f"f{i}" for i in range(Xtr.shape[1])]))
    return models

if RUN_SR:
    sr = cache("linear+SR", fit_linear_sr)
    record("linear+SR", linear.predict(Xva) + np.stack([m.predict(Xva) for m in sr], axis=1))
    for k, m in enumerate(sr):
        print(f"  mode {k}: residual = {m.get_best()['equation']}")

## Zernike representation of the core signals

The core signals, placed at their cores on the unit disk, projected onto the
first `n_zern` Noll Zernikes (36 for 19 cores; 10 for 7 cores, the fewest with
full rank on one hexagonal ring, where coma coincides with tilt). The sampled
basis has rank n_cores, so this is a change of variables rather than a
compression, shown as a diagnostic of how the mean core signal decomposes.


In [ ]:
# Zernike basis of the cores: a diagnostic of how the mean core signal decomposes
N_ZERN = S["n_zern"]
B = ml.zernike_basis(ds.core_xy, N_ZERN, fill=0.8)[0]
P = ml.modal_projector(B)
print(f"Zernike basis {B.shape}, rank {np.linalg.matrix_rank(B)}")

s_mean = S_tr.mean(axis=0)
xy = ds.core_xy / np.linalg.norm(ds.core_xy, axis=1).max()
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, s, title in [(ax[0], s_mean, "mean core signal"), (ax[1], B @ (P @ s_mean), "Zernike reconstruction")]:
    sc = a.scatter(xy[:, 0], xy[:, 1], c=s, s=350)
    for i, (x, y) in enumerate(xy):
        a.text(x, y, str(i), ha="center", va="center", fontsize=7, color="w")
    a.set_aspect("equal"); a.set_title(title); plt.colorbar(sc, ax=a)
plt.tight_layout(); plt.show()


## Compact model: OMP with a term budget K

Orthogonal Matching Pursuit keeps at most K of the degree-3 monomials of the
standardised core signals per mode (after screening them by correlation with
the mode). The sweep shows the accuracy / size trade-off.
Each piston has two OMP fits, one for its sine and one for its cosine; K is
the budget of each.


**Choosing the OMP budget K, per lantern.**

- *19 cores* (1539 degree-3 monomials). On `pl19_800` the sweep falls smoothly
  -- 15.5 nm at K = 1100, 9.3 nm at K = 1275 -- and stops at **K = 1281**,
  where every piston keeps its own natural support (1280 or 1281 monomials);
  every larger budget, up to the full cubic basis, reproduces that fit to the
  last digit (8.41 nm). K = 1282 is that saturated model counted with its
  intercept: its degrees of freedom per piston. `pl19_1200` uses the same
  budget; its own sweep shows whether it saturates at the same point.
- *7 cores* (119 monomials). On `pl7_1200` the sweep is nearly flat (from
  ~250 nm at K = 5 to ~224 nm with the complete cubic, K = 119) and there is
  no saturation point to find; the floor is the ambiguity of 7 signals, not
  the model, so K = 119 is used.


In [ ]:
SCREEN = 9000
KS = S["omp_ks"]
RECORD_KS = S["omp_record"]  # the budget chosen from the sweep (see the note above)

F3_tr, F3_va, _, poly_d3 = ml.poly_features(Xtr, Xva, 3)   # core-signal monomials
def omp_fits(K):
    return cache(f"fiber-omp-K{K}", lambda: [ml.omp_fit(F3_tr, Ytr[:, k], K, SCREEN)
                                             for k in range(Ytr.shape[1])])
def omp_predict(fits):
    return np.stack([ml.omp_predict(F3_va, f) for f in fits], axis=1)

sweep = {}
for K in KS:
    fits = omp_fits(K)
    C_hat = ys.inverse(omp_predict(fits))
    sweep[K] = (np.mean([ml.n_terms(f) for f in fits]), ml.rmse_nm(C_hat, C_va).mean(),
                ml.rmse_nm(C_hat, C_va, LAMBDA).mean())
    print(f"K={K:5d}: {sweep[K][0]:7.1f} terms/mode   mean RMSE {sweep[K][1]:5.1f} nm"
          f"   modulo lambda {sweep[K][2]:5.1f} nm")
for K in RECORD_KS:
    record(f"fiber-omp-K{K}", omp_predict(omp_fits(K)))

fig, ax = plt.subplots(figsize=(6, 4))
# the figure of merit: errors modulo lambda (with phasor targets an estimate across the
# wrap is the right piston, and the plain RMSE would count it as a ~lambda error)
ax.plot([v[0] for v in sweep.values()], [v[2] for v in sweep.values()], "ks-")
ax.set_yscale("log")
ax.set_xlabel("terms per fit (sin or cos of one piston)"); ax.set_ylabel("mean validation RMSE modulo lambda (nm)")
ax.set_title("core-signal degree-3 OMP: accuracy vs. model size"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Comparison

In [ ]:
order = ["linear", "linear+SR", *[f"fiber-omp-K{K}" for K in RECORD_KS], "MLP", "kNN"]
names = [m for m in order if m in RESULTS]
for m in names:
    print(f"  {m:18s} {RESULTS[m].mean():6.1f} nm   per mode {np.round(RESULTS[m], 1)}"
          f"   modulo lambda {WRAPPED[m].mean():6.1f} nm   per mode {np.round(WRAPPED[m], 1)}")

# k-NN is fitted and listed above but left out of the comparison plot; the bars
# are ordered by decreasing mean error and drawn on a linear scale
plot_names = [m for m in names if m != "kNN"]
plot_names.sort(key=lambda m: -RESULTS[m].mean())
x, w = np.arange(M_MODES), 0.8 / len(plot_names)
plt.figure(figsize=(9, 4.5))
for i, m in enumerate(plot_names):
    plt.bar(x + (i - len(plot_names) / 2 + 0.5) * w, RESULTS[m], w, label=m)
plt.yscale("linear"); plt.xticks(x); plt.xlabel("DM mode"); plt.ylabel("validation RMSE (nm)")
plt.title(f"Reconstructors, {LANTERN} (sorted by decreasing error)")
plt.legend(fontsize=8)
plt.grid(True, axis="y", ls=":", alpha=0.5); plt.tight_layout(); plt.show()

## Which monomials does OMP keep?

Kept vs. available monomials per degree at K = `k_table` (500 for 19 cores,
60 for 7), and how the degree mix
of the kept terms changes with the budget.

In [ ]:
K_TABLE, KS_MIX = S["k_table"], S["ks_mix"]
degree = poly_d3.powers_.sum(axis=1)
degs = np.arange(1, degree.max() + 1)
total = np.array([(degree == d).sum() for d in degs])
def kept_by_degree(K):
    kept = [f[1][np.flatnonzero(f[0].coef_)] for f in omp_fits(K)]
    return np.array([[np.sum(degree[g] == d) for d in degs] for g in kept])

kept = kept_by_degree(K_TABLE)
print(f"K = {K_TABLE}: kept / available monomials per degree (one column per mode)")
for i, d in enumerate(degs):
    print(f"  degree {d}: {total[i]:5d} available, kept {kept[:, i]}  ({kept[:, i].mean() / total[i]:.1%})")

mix = np.array([kept_by_degree(K).sum(axis=0) for K in KS_MIX], float)
mix = (mix / mix.sum(axis=1, keepdims=True)).T
plt.figure(figsize=(6, 4))
plt.stackplot(KS_MIX, *mix, labels=[f"degree {d}" for d in degs], alpha=0.85)
plt.xlabel("K (terms per mode)"); plt.ylabel("share of kept terms")
plt.title("degree mix of the kept terms"); plt.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

## Which core signals matter?

For the recorded OMP model: the total |coefficient| (in standardised units)
of the monomials each core signal appears in, per DM mode (core 0 is the
central one).


In [ ]:
K = S["k_importance"]
contains = (poly_d3.powers_ > 0).astype(float)      # monomial i involves variable v
vnames = [f"core {i}" for i in range(Xtr.shape[1])]
mass = np.zeros((M_MODES, len(vnames)))
for k, (omp, sel) in enumerate(omp_fits(K)):
    nz = np.flatnonzero(omp.coef_)
    mass[k % M_MODES] += contains[sel[nz]].T @ np.abs(omp.coef_[nz])   # sin and cos fits of piston k

top = np.argsort(-mass.sum(axis=0))[:20]
plt.figure(figsize=(10, 5))
bottom = np.zeros(len(top))
for k in range(M_MODES):
    plt.bar([vnames[i] for i in top], mass[k, top], bottom=bottom, label=f"piston {k}")
    bottom += mass[k, top]
plt.xticks(rotation=60); plt.ylabel("total |coefficient|")
plt.title(f"core-signal importance in fiber-omp-K{K}"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()


## Wavefront error besides the pistons

The PSF reaching the lantern is also affected by the wavefront errors of the optics. In the
ANDES error budget the optical surface errors and the NCPA follow a power spectral density
PSD(f) ~ f^-2 (30 nm rms for the optical surfaces; the NCPA split into ~11.9 nm rms of
low-order modes, up to spherical aberration, and ~27.5 nm rms beyond).

`gen_wfe.py` simulates the 10000 piston configurations of the single-wavelength dataset again,
each with its own f^-2 phase screen of 30, 60 or 90 nm rms (`ml_reconstruct.wavefront.psd_screens`:
global and petal pistons removed, so the truth stays the petal pistons; highest frequency
10 cycles per pupil, which reproduces the 11.9 / 27.5 nm split at 30 nm). Level 0 is the same
pipeline without a screen: the clean reference, on the same characterization.

Every model is **trained at the level it is tested at** (its own 8000 training rows, the 2000
validation rows of the same level); for reference, the models trained on the clean reference
are also applied to every level.

In [ ]:
from sklearn.linear_model import Ridge

WFE_LEVELS = [0, 30, 60, 90]                    # nm rms (gen_wfe.py); 0: no screen
WFE_KS = S["wfe_ks"]                            # OMP budgets scanned
DSW = {r: ml.LanternDataset.load(f"{DATA}/single_wfe{r:g}.npz") for r in WFE_LEVELS}

WFE = {}                                        # model -> {rms: mean RMSE modulo lambda [nm]}
def record_wfe(name, rms, C_hat):
    r = ml.rmse_nm(C_hat, C_va_w, LAMBDA).mean()
    WFE.setdefault(name, {})[rms] = r
    print(f"  WFE {rms:3g} nm  {name:26s} {r:9.3f} nm", flush=True)

def fit_all(tag, X_tr, Y_tr):
    """linear, dense quadratic, cubic OMP at each budget, MLP: all on phasor targets."""
    F_tr = ml.poly_features(X_tr, X_tr[:1], 3)[0]
    models = {"linear": cache(f"{tag}-linear", lambda: Ridge(1.0).fit(X_tr, Y_tr)),
              "dense quadratic": cache(f"{tag}-d2", lambda: ml.fit_dense_poly_ridge(X_tr, Y_tr, 2))}
    for K in WFE_KS:
        models[f"OMP cubic K={K}"] = cache(f"{tag}-omp-K{K}", lambda: [ml.omp_fit(F_tr, Y_tr[:, k], K, SCREEN)
                                                                   for k in range(Y_tr.shape[1])])
    path = f"{DATA}/cache_single_phasor/{tag}-mlp-{'x'.join(map(str, LAYERS))}-e500.npz"
    if os.path.exists(path):
        models["MLP"] = ml.load_mlp(path)[0]
    else:
        pm, hm = ml.train_mlp(LAYERS, X_tr, Y_tr, X_tr[:2000], Y_tr[:2000], epochs=500,
                              learning_rate=3e-4, log_every=1000)
        ml.save_mlp(path, pm, LAYERS, train=hm["train"], val=hm["val"], best_epoch=hm["best_epoch"])
        models["MLP"] = pm
    return models

def predict_all(models, X_tr, X):
    F = ml.poly_features(X_tr, X, 3)[1]
    out = {"linear": models["linear"].predict(X),
           "dense quadratic": ml.predict_dense_poly_ridge(models["dense quadratic"], X),
           "MLP": np.asarray(ml.mlp_forward(models["MLP"], X))}
    for K in WFE_KS:
        out[f"OMP cubic K={K}"] = np.stack([ml.omp_predict(F, f) for f in models[f"OMP cubic K={K}"]], axis=1)
    return {m: ys.inverse(Y) for m, Y in out.items()}

# the models of the clean reference (level 0), kept for the cross test
S0_tr, C0_tr = DSW[0].train
x0 = ml.Standardizer().fit(S0_tr)
X0_tr = x0.transform(S0_tr)
clean_models = fit_all("wfe0", X0_tr, ys.transform(C0_tr))

for rms in WFE_LEVELS:
    Sw_tr, Cw_tr = DSW[rms].train
    Sw_va, C_va_w = DSW[rms].val
    xw = ml.Standardizer().fit(Sw_tr)
    Xw_tr = xw.transform(Sw_tr)
    models = clean_models if rms == 0 else fit_all(f"wfe{rms:g}", Xw_tr, ys.transform(Cw_tr))
    for m, C_hat in predict_all(models, Xw_tr, xw.transform(Sw_va)).items():
        record_wfe(m, rms, C_hat)
    if rms:                                      # the clean-trained models on this level
        P0 = predict_all(clean_models, X0_tr, x0.transform(Sw_va))
        record_wfe(f"OMP K={max(WFE_KS)}, trained clean", rms, P0[f"OMP cubic K={max(WFE_KS)}"])
        record_wfe("MLP, trained clean", rms, P0["MLP"])

In [ ]:
omp_names = [f"OMP cubic K={K}" for K in WFE_KS]
WFE["OMP cubic, best K"] = {r: min(WFE[m][r] for m in omp_names) for r in WFE_LEVELS}
for name in ("OMP K=%d, trained clean" % max(WFE_KS), "MLP, trained clean"):
    WFE[name][0] = WFE["OMP cubic K=%d" % max(WFE_KS) if name.startswith("OMP") else "MLP"][0]
print("WFE rms (nm)              " + "".join(f"{r:>10g}" for r in WFE_LEVELS))
for m, r in WFE.items():
    print(f"{m:26s}" + "".join(f"{r[x]:10.3f}" for x in WFE_LEVELS))

plt.figure(figsize=(7, 4.5))
for m, style in [("linear", "s-"), ("dense quadratic", "^-"), ("OMP cubic, best K", "o-"), ("MLP", "D-"),
                 ("OMP K=%d, trained clean" % max(WFE_KS), "x:"), ("MLP, trained clean", "+:")]:
    plt.semilogy(WFE_LEVELS, [WFE[m][r] for r in WFE_LEVELS], style, label=m)
plt.xticks(WFE_LEVELS)
plt.xlabel("wavefront error besides the pistons (nm rms, PSD ~ f^-2)")
plt.ylabel("RMSE modulo lambda (nm)")
plt.title(f"{LANTERN}: single wavelength with a wavefront error, models trained at each level")
plt.grid(alpha=0.3, which="both"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()